# DiGToR — Phân tích lỗi: 2 ca chủ đạo (self-contained, chỉ cần Run All)

Notebook này **tự động** dựng đúng hai hình minh hoạ lỗi trong paper:

1. **Cross-over nhiệt trên FMB** — nền bị hâm nóng tới gần nhiệt vật thể: ảnh nhiệt mất
   tương phản biên nhưng `c_t` vẫn cao, router gửi pixel sang **nhánh Tin-nhiệt (T-rescue)**
   và dự đoán **sai**. (Hạn chế của reliability hiện tại — nhạy dropout/noise nhưng chưa
   nhạy mất tương phản cục bộ.)
2. **Định tuyến mặc định Visible trên SemanticRT** — vùng *easy* chiếm ~88% pixel và mục
   tiêu route gán mọi pixel V-đúng vào **V-trust**; dù **nhiệt là stream mạnh hơn**, phần lớn
   vùng easy vẫn bị đẩy về Visible → giảm clean mIoU.

**Run All** sẽ: clone repo → tải **FMB + SemanticRT** từ Google Drive → kéo checkpoint đã
train từ **Weights & Biases** (`digtor-fmb` *và* `digtor-semanticrt`) → quét test set tìm
ca tiêu biểu cho mỗi lỗi → in **panel so sánh** + chẩn đoán định lượng.

> Yêu cầu: cả hai wandb project đã có checkpoint (`v_only/t_only/fusion/digtor`). Trên
> Kaggle thêm secret `WANDB_API_KEY`; trên Colab cell wandb hỏi key một lần.

In [ ]:
import os
import shutil

repo_name = "DiGToR"

# Nếu thư mục đã tồn tại, xóa đi để chuẩn bị tải mới
if os.path.exists(repo_name):
    print(f"Phát hiện thư mục '{repo_name}' đã tồn tại. Đang xóa...")
    shutil.rmtree(repo_name)
    print("Đã xóa thư mục cũ.")

repo_url = f"https://github.com/nguyenmaiductrong/{repo_name}.git"

print(f"Đang tải repo từ {repo_url}...")
exit_code = os.system(f"git clone {repo_url}")

if exit_code == 0:
    print("Tải thành công. Kiểm tra ở thư mục Output/Working.")
else:
    print("Có lỗi xảy ra khi tải, kiểm tra lại đường dẫn mạng hoặc URL.")

Đang tải repo từ https://github.com/nguyenmaiductrong/DiGToR.git...
Tải thành công. Kiểm tra ở thư mục Output/Working.


In [ ]:
# repo da duoc clone moi o cell tren (fresh = latest main); pull nay chi de chac chan
!cd DiGToR && git pull origin main || echo "(pull skipped - fresh clone already latest)"

From https://github.com/nguyenmaiductrong/DiGToR
 * branch            main       -> FETCH_HEAD
Already up to date.


In [ ]:
cd DiGToR

/content/DiGToR


In [ ]:
# --- Download FMB data from Google Drive and unzip train/test ---
# Data is stored OUTSIDE the cloned repo so a re-clone does not wipe it, and the
# step is idempotent: re-running skips the ~1GB download if the data is ready.
import os, glob, zipfile, shutil, subprocess, sys

DRIVE_URL = "https://drive.google.com/drive/folders/1T_jVi80tjgyHTQDpn-TjfySyW4CK1LlF"
FMB_DIR = "/content/FMB" if os.path.isdir("/content") else os.path.join(
    os.path.dirname(os.getcwd()), "FMB")
_mods = ("Visible", "Infrared", "Label")

def _ready(split):
    return all(os.path.isdir(os.path.join(FMB_DIR, split, m)) for m in _mods)

if _ready("train") and _ready("test"):
    print("FMB already prepared at", FMB_DIR)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown
    dl = os.path.join(os.getcwd(), "_fmb_download")
    shutil.rmtree(dl, ignore_errors=True)
    os.makedirs(dl, exist_ok=True)
    gdown.download_folder(DRIVE_URL, output=dl, quiet=False, use_cookies=False)

    def _find(name):
        hits = glob.glob(os.path.join(dl, "**", name), recursive=True)
        if not hits:
            raise FileNotFoundError(f"{name} not found in the Drive folder")
        return hits[0]

    def _modroot(base):
        # the directory that directly holds Visible/Infrared/Label, whatever the
        # zip's internal nesting is
        for root, _, _ in os.walk(base):
            if all(os.path.isdir(os.path.join(root, m)) for m in _mods):
                return root
        raise FileNotFoundError(f"no Visible/Infrared/Label folder under {base}")

    os.makedirs(FMB_DIR, exist_ok=True)
    for split in ("train", "test"):
        tmp = os.path.join(os.getcwd(), f"_unzip_{split}")
        shutil.rmtree(tmp, ignore_errors=True)
        print(f"unzipping {split}.zip ...")
        with zipfile.ZipFile(_find(f"{split}.zip")) as z:
            z.extractall(tmp)
        dst = os.path.join(FMB_DIR, split)
        shutil.rmtree(dst, ignore_errors=True)
        shutil.move(_modroot(tmp), dst)
        shutil.rmtree(tmp, ignore_errors=True)
    shutil.rmtree(dl, ignore_errors=True)
    print("Prepared FMB at", FMB_DIR)

# The data-detection cell below reads FMB_ROOT first, so this works regardless of
# where the data was staged.
os.environ["FMB_ROOT"] = FMB_DIR
for split in ("train", "test"):
    n = len(glob.glob(os.path.join(FMB_DIR, split, "Label", "*"))) if _ready(split) else 0
    print(f"  {split}: {n} label files")

# --- strip macOS AppleDouble junk (._*) + __MACOSX that break PIL.open ---
import subprocess as _sp
if FMB_DIR and os.path.isdir(FMB_DIR):
    _sp.run(['bash','-lc',"find '"+FMB_DIR+"' -name '._*' -delete; rm -rf '"+FMB_DIR+"/__MACOSX'"])
    print('cleaned AppleDouble junk under', FMB_DIR)


Retrieving folder contents


Processing file 1AamLVgMG5JCkhUOZi-xKTymZXCNklnaq test.zip
Processing file 1nk2R2yZ05SDe2lDpsk2D5RAngMBPUPsB train.zip


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1AamLVgMG5JCkhUOZi-xKTymZXCNklnaq
From (redirected): https://drive.google.com/uc?id=1AamLVgMG5JCkhUOZi-xKTymZXCNklnaq&confirm=t&uuid=ffdc67a1-dec3-4e99-94ad-153e746fd6f3
To: /content/DiGToR/_fmb_download/test.zip
100%|██████████| 216M/216M [00:06<00:00, 32.9MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1nk2R2yZ05SDe2lDpsk2D5RAngMBPUPsB
From (redirected): https://drive.google.com/uc?id=1nk2R2yZ05SDe2lDpsk2D5RAngMBPUPsB&confirm=t&uuid=161c89d0-5e10-4e14-b136-d8e434700b1a
To: /content/DiGToR/_fmb_download/train.zip
100%|██████████| 914M/914M [00:10<00:00, 88.2MB/s]
Download completed


unzipping train.zip ...
unzipping test.zip ...
Prepared FMB at /content/FMB
  train: 1220 label files
  test: 280 label files


In [ ]:
# --- Download SemanticRT from Google Drive (mirrors digtor-semanticrt.ipynb) ---
# IMPORTANT: the moved folder must contain rgb/thermal/labels AND train/val/test.txt,
# otherwise digtor's loader falls back to a block-shuffle split (the bug that gave
# "test images: 0" / picked up AppleDouble ._ files). We require the .txt files here.
import os, glob, zipfile, shutil, subprocess, sys

SEMRT_FID = "1KXgqYLy-yQBLAzhXY1zusVQiQ27Gr0Uw"   # same id as digtor-semanticrt.ipynb
SEMRT_DIR = "/content/SemanticRT_dataset" if os.path.isdir("/content") else \
    os.path.join(os.path.dirname(os.getcwd()), "SemanticRT_dataset")
_smods = ("rgb", "thermal", "labels")
_ssplits = ("train.txt", "val.txt", "test.txt")

def _semrt_ready(base):
    return (all(os.path.isdir(os.path.join(base, m)) for m in _smods)
            and all(os.path.isfile(os.path.join(base, s)) for s in _ssplits))

def _semrt_dataroot(base):
    # the directory that DIRECTLY holds rgb/thermal/labels + the split txt files,
    # whatever the zip's internal nesting is
    for root, _, _ in os.walk(base):
        if _semrt_ready(root):
            return root
    raise FileNotFoundError(f"no rgb/thermal/labels(+train/val/test.txt) under {base}")

if _semrt_ready(SEMRT_DIR):
    print("SemanticRT already prepared at", SEMRT_DIR)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown
    dl = os.path.join(os.getcwd(), "_semrt_download")
    shutil.rmtree(dl, ignore_errors=True); os.makedirs(dl, exist_ok=True)
    zpath = os.path.join(dl, "semanticrt.zip")
    gdown.download(id=SEMRT_FID, output=zpath, quiet=False)
    tmp = os.path.join(os.getcwd(), "_semrt_unzip"); shutil.rmtree(tmp, ignore_errors=True)
    print("unzipping semanticrt.zip ...")
    with zipfile.ZipFile(zpath) as z:
        z.extractall(tmp)
    shutil.rmtree(SEMRT_DIR, ignore_errors=True)
    shutil.move(_semrt_dataroot(tmp), SEMRT_DIR)   # <- requires the .txt files
    shutil.rmtree(tmp, ignore_errors=True); shutil.rmtree(dl, ignore_errors=True)
    print("Prepared SemanticRT at", SEMRT_DIR)

# strip macOS AppleDouble junk (._*) + __MACOSX (harmless once the official split is used)
subprocess.run(["bash", "-lc", f"find '{SEMRT_DIR}' -name '._*' -delete; rm -rf '{SEMRT_DIR}/__MACOSX'"])

os.environ["SEMRT_ROOT"] = SEMRT_DIR
from digtor.dataset.semanticrt import has_official_split
_n_test = sum(1 for _ in open(os.path.join(SEMRT_DIR, "test.txt"))) if _semrt_ready(SEMRT_DIR) else 0
print("SEMRT_ROOT =", SEMRT_DIR, "| ready:", _semrt_ready(SEMRT_DIR),
      "| official split:", has_official_split(SEMRT_DIR), "| test.txt lines:", _n_test)
assert _semrt_ready(SEMRT_DIR), "SemanticRT missing rgb/thermal/labels or train/val/test.txt"


In [ ]:
# --- Repo + FMB data (auto-detect official train/test split OR legacy flat) ---
# Runs on Colab (Pro / A100) or Kaggle. The previous cell downloads the data and
# sets FMB_ROOT, which is checked first below; the other paths are fallbacks.
import os, sys, glob

REPO = os.getcwd()                 # current dir = the DiGToR repo
assert os.path.isfile(os.path.join(REPO, 'digtor', '__init__.py')), \
    f'Run this from the DiGToR repo root (no digtor/ package found in {REPO}).'

_mods = ['Visible', 'Infrared', 'Label']

def _is_fmb_root(path):
    flat = all(os.path.isdir(os.path.join(path, s)) for s in _mods)
    split = all(os.path.isdir(os.path.join(path, sp, s)) for sp in ['train', 'test'] for s in _mods)
    return flat or split, split

# Priority: FMB_ROOT (set by the download cell) -> repo/FMB -> Colab /content or
# Drive -> Kaggle inputs.
_candidates = []
if os.environ.get('FMB_ROOT'):
    _candidates.append(os.environ['FMB_ROOT'])
_candidates.append(os.path.join(REPO, 'FMB'))
_candidates += ['/content/FMB', '/content/drive/MyDrive/FMB']
_candidates += glob.glob('/kaggle/input/**/FMB', recursive=True)
_candidates += glob.glob('/kaggle/input/*', recursive=False)

ROOT = None
SPLIT_LAYOUT = False
for _cand in dict.fromkeys(_candidates):
    ok, split = _is_fmb_root(_cand)
    if ok:
        ROOT, SPLIT_LAYOUT = _cand, split
        break
assert ROOT is not None, 'FMB data not found: need flat Visible/Infrared/Label OR official train/test layout.'

print('FMB layout =', 'official train/test (held-out test)' if SPLIT_LAYOUT else 'legacy flat (auto-split)')
print('REPO =', REPO)
print('ROOT =', ROOT)

sys.path.insert(0, REPO)
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

FMB layout = official train/test (held-out test)
REPO = /content/DiGToR
ROOT = /content/FMB
CUDA: True NVIDIA L4


In [ ]:
# --- Weights & Biases: pull the already-trained checkpoints (no retrain) ---
# Logs into wandb and pulls each <mode>-ckpt artifact into CKPT. Set USE_WANDB=False
# only if the checkpoints are already staged locally in ckpt_fmb/.
import os, subprocess, sys

USE_WANDB = True
WB_PROJECT = 'digtor-fmb'
WB_ENTITY = None            # None = your default wandb entity
CKPT = 'ckpt_fmb'
os.makedirs(CKPT, exist_ok=True)

if USE_WANDB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wandb'], check=True)
    import wandb
    # Kaggle: add your key as a secret named WANDB_API_KEY. Colab: this prompts
    # once, or set os.environ['WANDB_API_KEY'] = '...' before running.
    try:
        wandb.login()
    except Exception as e:
        print('wandb.login failed -> disabling wandb:', e); USE_WANDB = False

ENTITY_ARG = f'--entity {WB_ENTITY}' if WB_ENTITY else ''
print('wandb:', 'ON' if USE_WANDB else 'OFF', '| CKPT =', CKPT)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nguyenmaiductrong37 (nguyenmaiductrong37-h-c-vi-n-c-ng-ngh-b-u-ch-nh-vi-n-th-ng) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: ON | CKPT = ckpt_fmb


In [ ]:
# --- Pull already-trained checkpoints for BOTH datasets (inference only, no retrain) ---
# Idempotent + failsafe. FMB -> ckpt_fmb (project digtor-fmb); SemanticRT -> ckpt_semanticrt
# (project digtor-semanticrt). Missing artifacts raise below so the figures don't run blind.
import os as _os

CKPT_FMB, CKPT_SRT = "ckpt_fmb", "ckpt_semanticrt"
if USE_WANDB:
    !python -m digtor.wandb_ckpt --dataset fmb        --pull --project digtor-fmb        {ENTITY_ARG} --out {CKPT_FMB} --modes v_only t_only fusion digtor
    !python -m digtor.wandb_ckpt --dataset semanticrt --pull --project digtor-semanticrt {ENTITY_ARG} --out {CKPT_SRT} --modes v_only t_only fusion digtor
else:
    print("wandb OFF -> expecting checkpoints already in", CKPT_FMB, "and", CKPT_SRT)

for ck in (CKPT_FMB, CKPT_SRT):
    have = [n for n in ["v_only", "t_only", "fusion", "digtor"]
            if _os.path.isfile(_os.path.join(ck, n + ".pt"))]
    print(f"{ck}: {have}")
    assert len(have) == 4, f"missing checkpoints in {ck}: need all 4, got {have}. Train first (with --wandb) or stage them."

## Phân tích — nạp model, helper dùng chung

Suy luận **đúng đường eval**: định tuyến cứng `hard=True`, `rel_gate=0`, bỏ nền (`ignore_bg`)
nên khớp số four-region trong paper. Hai bộ model (FMB / SemanticRT) nạp riêng.

In [ ]:
# --- Shared imports, palettes, model loader, inference + map helpers ---
import os, sys
import numpy as np, torch, torch.nn.functional as F
import matplotlib.pyplot as plt

from digtor import (IGNORE_INDEX, DATASET_CONFIGS, FMB_CLASS_NAMES, SEMRT_CLASS_NAMES)
from digtor.models import build_model
from digtor.metrics import four_region_partition, correctness_mask
from digtor.dataset.fmb import IMAGENET_MEAN, IMAGENET_STD

H, W, BASE = 384, 512, 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REPO = globals().get("REPO") or os.getcwd()
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# routing colours = demo/app.py: V-trust blue, T-rescue red, Joint green.
ROUTE_COLORS = np.array([(66, 135, 245), (240, 90, 60), (120, 200, 90)], np.uint8)
ROUTE_NAMES = ["V-trust", "T-rescue", "Joint"]

def make_palette(nc):
    pal = (plt.get_cmap("tab20")(np.arange(nc) % 20)[:, :3] * 255).astype(np.uint8)
    pal[0] = (0, 0, 0)                      # class 0 = unlabelled -> black
    return pal

def colorize(label, pal):
    return pal[np.clip(label, 0, len(pal) - 1)]

def denorm_rgb(t):
    a = t.cpu().numpy().transpose(1, 2, 0) * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(a, 0, 1)

def ir_to_img(t):
    return np.clip(t.cpu().numpy()[0] * 0.5 + 0.5, 0, 1)

def load_models(ckpt_dir, nc):
    ms = {}
    for n in ["v_only", "t_only", "fusion", "digtor"]:
        m = build_model(n, base=BASE, num_classes=nc).to(DEVICE).eval()
        sd = torch.load(os.path.join(ckpt_dir, n + ".pt"), map_location=DEVICE)["model"]
        m.load_state_dict(sd, strict=False)
        ms[n] = m
    return ms

@torch.no_grad()
def infer(models, item):
    """One image -> preds (4 models), full-res hard route, full-res cV/cT maps + scalars."""
    rgb = item["rgb"][None].to(DEVICE); ir = item["ir"][None].to(DEVICE)
    gt = item["label"].numpy()
    preds = {
        "v_only": models["v_only"](rgb).argmax(1)[0].cpu().numpy(),
        "t_only": models["t_only"](ir).argmax(1)[0].cpu().numpy(),
        "fusion": models["fusion"](rgb, ir).argmax(1)[0].cpu().numpy(),
    }
    ld, aux = models["digtor"](rgb, ir, hard=True, return_aux=True, rel_gate=0)
    preds["digtor"] = ld.argmax(1)[0].cpu().numpy()
    route = F.interpolate(aux["pi"], size=gt.shape, mode="nearest").argmax(1)[0].cpu().numpy()
    up = lambda m: F.interpolate(m, size=gt.shape, mode="bilinear", align_corners=False)[0, 0].cpu().numpy()
    return gt, preds, route, up(aux["ct"]), up(aux["cv"]), float(aux["cv"].mean()), float(aux["ct"].mean())

def error_map(pred, gt_r):
    valid = gt_r != IGNORE_INDEX
    out = np.zeros((*gt_r.shape, 3), np.uint8)
    out[valid] = (40, 160, 40)                    # correct = green
    out[(pred != gt_r) & valid] = (220, 30, 30)   # wrong = red
    return out

def route_map(route, gt_r):
    out = ROUTE_COLORS[np.clip(route, 0, 2)].copy()
    out[gt_r == IGNORE_INDEX] = (30, 30, 30)      # ignored background = grey
    return out

def route_share(route, gt_r):
    valid = gt_r != IGNORE_INDEX
    r = route[valid]
    return np.array([(r == k).mean() for k in range(3)]) if r.size else np.zeros(3)

# thu muc luu anh slide: uu tien /content (Colab), fallback cwd
FIG_DIR = "/content/digtor_error_figs" if os.path.isdir("/content") else os.path.join(os.getcwd(), "digtor_error_figs")
os.makedirs(FIG_DIR, exist_ok=True)

print("helpers ready | device:", DEVICE, "| FIG_DIR:", FIG_DIR)

In [ ]:
# === HÌNH 1 — Cross-over nhiệt trên FMB ===========================================
# Tìm ca: pixel được route sang nhánh Tin-nhiệt (T-rescue), c_t CAO, nhưng nhiệt
# (và DiGToR) dự đoán SAI -> đúng định nghĩa thermal cross-over trong paper.
from digtor.dataset.fmb import FMBDataset, deterministic_split as fmb_split

NC_FMB = DATASET_CONFIGS["fmb"].num_classes
PAL_FMB = make_palette(NC_FMB)
fmb_models = load_models("ckpt_fmb", NC_FMB)
fmb_root = globals().get("ROOT") or os.environ.get("FMB_ROOT")
_fmb_names = [n for n in fmb_split(fmb_root)["test"] if not os.path.basename(str(n)).startswith("._")]
fmb_test = FMBDataset(fmb_root, _fmb_names, size=(H, W), partition="test")
print("FMB test images:", len(fmb_test))

best = None  # (score, idx, n_cross, ct_in, t_to_branch)
for idx in range(len(fmb_test)):
    gt, preds, route, ct_map, cv_map, cv, ct = infer(fmb_models, fmb_test[idx])
    gt_r = gt.copy(); gt_r[gt_r == 0] = IGNORE_INDEX
    valid = gt_r != IGNORE_INDEX; nval = max(int(valid.sum()), 1)
    Tbranch = (route == 1)
    crossover = Tbranch & (preds["digtor"] != gt_r) & (preds["t_only"] != gt_r) & valid
    n_cross = int(crossover.sum())
    if n_cross == 0:
        continue
    ct_in = float(ct_map[crossover].mean())                 # confidence in the misleading region
    score = (n_cross / nval) * ct_in                        # large + high-confidence wrong T-branch
    if best is None or score > best[0]:
        best = (score, idx, n_cross, ct_in, float((Tbranch & valid).sum()) / nval)

assert best is not None, "no thermal cross-over case found on FMB test"
_, idx, n_cross, ct_in, t_share = best
item = fmb_test[idx]
gt, preds, route, ct_map, cv_map, cv, ct = infer(fmb_models, item)
gt_r = gt.copy(); gt_r[gt_r == 0] = IGNORE_INDEX
valid = gt_r != IGNORE_INDEX
wrong_T = (route == 1) & (preds["digtor"] != gt_r) & valid

# --- 1 ảnh slide-ready: 7 panel + tiêu đề + hộp ghi chú nhúng sẵn ---
pct_wrongT = 100 * int(wrong_T.sum()) / max(int(valid.sum()), 1)
panels = [("RGB", denorm_rgb(item["rgb"]), None),
          ("Thermal (IR)", ir_to_img(item["ir"]), "gray"),
          ("Ground truth", colorize(gt, PAL_FMB), None),
          ("DiGToR pred", colorize(preds["digtor"], PAL_FMB), None),
          ("Error (đỏ = sai)", error_map(preds["digtor"], gt_r), None),
          ("Routing (lam=V, đỏ=T, lục=J)", route_map(route, gt_r), None),
          ("c_t (tin cậy nhiệt)", ct_map, "inferno")]
fig, axes = plt.subplots(2, 4, figsize=(18, 9.8))
ax = axes.ravel(); NCOL = 4
for j, (t, im, cm) in enumerate(panels):
    a = ax[j]
    a.imshow(im, cmap=cm, vmin=0 if cm == "inferno" else None, vmax=1 if cm == "inferno" else None)
    a.axis("off")
    if j // NCOL == 0:                       # hàng đầu -> nhãn ở TRÊN
        a.set_title(t, fontsize=11)
    else:                                     # hàng 2 -> nhãn ở DƯỚI
        a.text(0.5, -0.06, t, transform=a.transAxes, ha="center", va="top", fontsize=11)
for a in ax[len(panels):]:
    a.axis("off")  # ẩn ô thừa (vị trí thứ 8)
fig.suptitle("Hình 1 · Cross-over nhiệt trên FMB — c_t cao nhưng nhánh Tin-nhiệt dự đoán SAI"
             f"   (ảnh test [{idx}] {item['name']}, điều kiện: {item['cond']})",
             fontsize=15, fontweight="bold", y=0.97)
note = (
    "CƠ CHẾ: nền bị hâm nóng tới gần nhiệt vật thể → ảnh nhiệt mất TƯƠNG PHẢN BIÊN nhưng c_t vẫn CAO "
    "→ router gửi pixel sang nhánh T-rescue (đỏ ở Routing) và dự đoán SAI (đỏ ở Error).\n"
    f"SỐ LIỆU: {int(wrong_T.sum())} px route sang T-rescue nhưng sai ({pct_wrongT:.1f}% pixel hợp lệ)  ·  "
    f"c_t trung bình tại vùng sai = {ct_in:.2f} (cao)  ·  {t_share*100:.1f}% pixel đi vào nhánh nhiệt.\n"
    "Ý NGHĨA: reliability học từ đúng/sai teacher bắt được dropout/noise nhưng CHƯA nhạy với mất tương phản "
    "cục bộ → hướng sửa: thêm mục tiêu tin cậy nhận biết tương phản / đặc trưng biên nhiệt."
)
fig.text(0.5, 0.07, note, ha="center", va="center", fontsize=12.5, wrap=True,
         bbox=dict(boxstyle="round,pad=0.6", fc="#fff7e0", ec="#e0a000", lw=1.2))
fig.tight_layout(rect=[0, 0.16, 1, 0.94])
_out1 = os.path.join(FIG_DIR, "fig13_fmb_thermal_crossover.png")
fig.savefig(_out1, dpi=180, bbox_inches="tight")
plt.show()
print("Đã lưu ảnh slide (kèm tiêu đề + ghi chú) ->", _out1)


## Hình 2 — Định tuyến mặc định Visible trên SemanticRT

Trên SemanticRT vùng *easy* (~88% pixel) bị route mặc định về **V-trust** dù **nhiệt là
stream mạnh hơn**. Notebook quét test set và chọn ca: vùng easy lớn, tỉ lệ route V-trust cao,
trong khi T-only đúng ≥ V-only (chứng tỏ nhiệt mạnh hơn nhưng vẫn bị bỏ qua).

In [ ]:
# === HÌNH 2 — Default-Visible làm DiGToR DỰ ĐOÁN SAI trên SemanticRT ==============
# Chọn ca: pixel mà NHIỆT đúng nhưng RGB sai, router vẫn mặc định route về V-trust
# -> DiGToR dự đoán SAI. Đây là cái giá NHÌN-THẤY-ĐƯỢC (per-pixel) của thiên lệch
# Visible, đối xứng với cross-over nhiệt của FMB (route T nhưng sai).
from digtor.dataset.semanticrt import SemanticRTDataset, deterministic_split as srt_split
from digtor.dataset.semanticrt import has_official_split

NC_SRT = DATASET_CONFIGS["semanticrt"].num_classes
PAL_SRT = make_palette(NC_SRT)
srt_models = load_models("ckpt_semanticrt", NC_SRT)
srt_root = os.environ.get("SEMRT_ROOT") or globals().get("SEMRT_DIR")
assert srt_root and has_official_split(srt_root), \
    f"SemanticRT root thieu official split (rgb/thermal/labels + train/val/test.txt): {srt_root}. Chay lai cell tai SemanticRT."
srt_test = SemanticRTDataset(srt_root, srt_split(srt_root)["test"], size=(H, W))
SCAN = min(len(srt_test), 600)
print("SemanticRT test images:", len(srt_test), "| scanning first", SCAN)

best = None  # (n_miss, idx, sh, t_acc, v_acc)
for idx in range(SCAN):
    gt, preds, route, ct_map, cv_map, cv, ct = infer(srt_models, srt_test[idx])
    gt_r = gt.copy(); gt_r[gt_r == 0] = IGNORE_INDEX
    valid = gt_r != IGNORE_INDEX; nval = max(int(valid.sum()), 1)
    # nhiệt-đúng, V-sai, route về V-trust (0), và DiGToR sai -> default-Visible gây lỗi
    miss = ((route == 0) & (preds["v_only"] != gt_r) & (preds["t_only"] == gt_r)
            & (preds["digtor"] != gt_r) & valid)
    n_miss = int(miss.sum())
    if n_miss == 0:
        continue
    sh = route_share(route, gt_r)
    t_acc = correctness_mask(preds["t_only"], gt_r)[valid].mean()
    v_acc = correctness_mask(preds["v_only"], gt_r)[valid].mean()
    if best is None or n_miss > best[0]:
        best = (n_miss, idx, sh, float(t_acc), float(v_acc))

assert best is not None, ("khong tim thay pixel (nhiet-dung, V-sai) bi route ve V va DiGToR sai "
                          "trong SCAN dau; tang SCAN len.")
_, idx, sh, t_acc, v_acc = best
item = srt_test[idx]
gt, preds, route, ct_map, cv_map, cv, ct = infer(srt_models, item)
gt_r = gt.copy(); gt_r[gt_r == 0] = IGNORE_INDEX
valid = gt_r != IGNORE_INDEX; nval = max(int(valid.sum()), 1)
miss = ((route == 0) & (preds["v_only"] != gt_r) & (preds["t_only"] == gt_r)
        & (preds["digtor"] != gt_r) & valid)
n_miss = int(miss.sum()); pct_miss = 100 * n_miss / nval

# --- 1 ảnh slide-ready (2x4): hàng đầu nhãn ở trên, hàng 2 nhãn ở dưới ---
panels = [("RGB", denorm_rgb(item["rgb"]), None),
          ("Thermal (IR)", ir_to_img(item["ir"]), "gray"),
          ("Ground truth", colorize(gt, PAL_SRT), None),
          ("DiGToR pred", colorize(preds["digtor"], PAL_SRT), None),
          ("Error (đỏ = SAI so với GT)", error_map(preds["digtor"], gt_r), None),
          ("Routing (lam=V, đỏ=T, lục=J)", route_map(route, gt_r), None)]
fig, axes = plt.subplots(2, 4, figsize=(18, 9.8))
ax = axes.ravel(); NCOL = 4
for j, (t, im, cm) in enumerate(panels):
    a = ax[j]
    a.imshow(im, cmap=cm); a.axis("off")
    if j // NCOL == 0:
        a.set_title(t, fontsize=11)
    else:
        a.text(0.5, -0.06, t, transform=a.transAxes, ha="center", va="top", fontsize=11)
# ô thứ 7 (bottom-right trừ ô ẩn): biểu đồ route share
ax[6].bar(ROUTE_NAMES, sh * 100, color=[tuple(c / 255 for c in ROUTE_COLORS[k]) for k in range(3)])
ax[6].set_ylabel("Route share (%)", fontsize=11); ax[6].set_ylim(0, 100)
for k, v in enumerate(sh * 100):
    ax[6].text(k, v + 1, f"{v:.0f}%", ha="center", fontsize=10)
ax[7].axis("off")  # ẩn ô thừa

fig.suptitle("Hình 2 · Default-Visible trên SemanticRT — pixel NHIỆT-đúng/RGB-sai bị route về V → DiGToR SAI"
             f"   (ảnh test [{idx}] {item['name']}, điều kiện: {item['cond']})",
             fontsize=14, fontweight="bold", y=0.97)
note = (
    "CƠ CHẾ: mục tiêu route 'mọi pixel V-đúng → V-trust' còn KÉO cả những pixel mà NHIỆT đúng nhưng RGB sai "
    "về nhánh V → DiGToR dự đoán SAI (vùng đỏ ở Error trùng vùng lam ở Routing).\n"
    f"SỐ LIỆU: {n_miss} px (nhiệt-đúng, RGB-sai) bị route về V-trust và DiGToR sai ({pct_miss:.1f}% pixel hợp lệ)  ·  "
    f"Route share V-trust {sh[0]*100:.1f}% · T-rescue {sh[1]*100:.1f}% · Joint {sh[2]*100:.1f}%  ·  T-only {t_acc*100:.1f}% ≥ V-only {v_acc*100:.1f}%.\n"
    "Ý NGHĨA: nhiệt là stream mạnh hơn nhưng default-Visible bỏ phí — vừa SAI từng pixel ở đây, vừa kéo clean mIoU "
    "toàn tập xuống (DiGToR 0.7675 < T-only 0.7794 < fusion 0.7843). Hướng sửa: route ĐỐI XỨNG modality "
    "(easy/rescue → stream mạnh hơn hoặc Joint, thay vì mặc định Visible)."
)
fig.text(0.5, 0.07, note, ha="center", va="center", fontsize=12.5, wrap=True,
         bbox=dict(boxstyle="round,pad=0.6", fc="#e8f0ff", ec="#3a6ec0", lw=1.2))
fig.tight_layout(rect=[0, 0.16, 1, 0.94])
_out2 = os.path.join(FIG_DIR, "fig_semrt_default_visible.png")
fig.savefig(_out2, dpi=180, bbox_inches="tight")
plt.show()
print("DIAGNOSTIC (SemanticRT default-Visible gây lỗi):")
print(f"  - {n_miss} px nhiệt-đúng/RGB-sai bị route về V-trust và DiGToR SAI ({pct_miss:.1f}% pixel hợp lệ)")
print(f"  - Route share: V {sh[0]*100:.1f}% | T {sh[1]*100:.1f}% | J {sh[2]*100:.1f}%  ·  T-only {t_acc*100:.1f}% ≥ V-only {v_acc*100:.1f}%")
print("Đã lưu ảnh slide (kèm tiêu đề + ghi chú) ->", _out2)


## Ghép vào báo cáo

| Hình | Quan sát trên panel | Nguyên nhân (vật lý / dữ liệu) |
| --- | --- | --- |
| **1. Cross-over nhiệt (FMB)** | Vùng route đỏ (T-rescue) trùng vùng lỗi đỏ; bản đồ `c_t` vẫn sáng (cao) ở đó | Nền hâm nóng tới gần nhiệt vật thể → nhiệt mất **tương phản biên** nhưng giữ cường độ; reliability học từ đúng/sai teacher bắt được dropout/noise nhưng **chưa nhạy mất tương phản cục bộ**. Hướng sửa: thêm mục tiêu tin cậy nhận biết tương phản / đặc trưng biên nhiệt. |
| **2. Mặc định Visible (SemanticRT)** | Routing map ngập **xanh (V-trust)**; bar route share V≫T,J; T-only đúng ≥ V-only | Vùng *easy* ~88% pixel + mục tiêu route gán mọi pixel V-đúng vào Visible-trust → đẩy phần lớn easy về **stream yếu hơn**, giảm clean mIoU. Hướng sửa: mục tiêu route **đối xứng modality** (easy → stream mạnh hơn hoặc Joint, thay vì đóng cứng Visible). |

Hai file ảnh đã lưu ở thư mục làm việc: `fig13_fmb_thermal_crossover.png` và `fig_semrt_default_visible.png`.